In [1]:
# CELLULE 1 - Configuration
from pyspark.sql import functions as F
from pyspark.sql.types import *

storage_account = "energybigdatastorage"
container_raw = "raw"
container_processed = "processed"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/daily_dataset/daily_dataset"
path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/daily_dataset/"

print(f"Source: {path_raw}")
print(f"Destination: {path_processed}")

In [3]:
# CELLULE 2 - Lecture avec schéma explicite
daily_schema = StructType([
    StructField("LCLid", StringType(), True),
    StructField("day", DateType(), True),
    StructField("energy_median", DoubleType(), True),
    StructField("energy_mean", DoubleType(), True),
    StructField("energy_max", DoubleType(), True),
    StructField("energy_count", IntegerType(), True),
    StructField("energy_std", DoubleType(), True),
    StructField("energy_sum", DoubleType(), True),
    StructField("energy_min", DoubleType(), True)
])

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(daily_schema) \
    .load(path_raw)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")
print("\n=== SCHEMA ===")
df.printSchema()
print("\n=== APERÇU ===")
df.show(5)

In [4]:
# CELLULE 3 - Statistiques avant nettoyage
print("\n=== VALEURS NULLES AVANT ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print("\n=== STATISTIQUES ===")
df.describe().show()

print("\n=== DOUBLONS ===")
print(f"Nombre de doublons : {df.count() - df.dropDuplicates().count()}")

In [5]:
# CELLULE 4 - Nettoyage
# 1. Supprimer lignes sans LCLid
df_clean = df.filter(F.col("LCLid").isNotNull())

# 2. Trim LCLid
df_clean = df_clean.withColumn("LCLid", F.trim(F.col("LCLid")))

# 3. Supprimer doublons
df_clean = df_clean.dropDuplicates()

# 4. Valeurs négatives -> NULL
energy_cols = ["energy_median", "energy_mean", "energy_max", "energy_sum", "energy_min"]
for c in energy_cols:
    df_clean = df_clean.withColumn(c, F.when(F.col(c) < 0, F.lit(None)).otherwise(F.col(c)))

# 5. NaN -> NULL
df_clean = df_clean.replace(float('nan'), None)

# 6. Supprimer lignes sans aucune donnée énergie
df_clean = df_clean.filter(
    F.col("energy_median").isNotNull() | 
    F.col("energy_mean").isNotNull() | 
    F.col("energy_sum").isNotNull()
)

# 7. Outliers IQR sur energy_mean
quantiles = df_clean.approxQuantile("energy_mean", [0.25, 0.75], 0.01)
if len(quantiles) == 2:
    q1, q3 = quantiles
    iqr = q3 - q1
    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr
    print(f"IQR bounds: [{lower:.4f}, {upper:.4f}]")
    df_clean = df_clean.withColumn("is_outlier_mean",
        F.when((F.col("energy_mean") < lower) | (F.col("energy_mean") > upper), F.lit(1)).otherwise(F.lit(0))
    )
else:
    df_clean = df_clean.withColumn("is_outlier_mean", F.lit(0))

# 8. Consistance max >= mean >= min
df_clean = df_clean.withColumn("consistency_flag",
    F.when(
        (F.col("energy_max").isNotNull()) & (F.col("energy_mean").isNotNull()) & (F.col("energy_min").isNotNull()) &
        ((F.col("energy_max") < F.col("energy_mean")) | (F.col("energy_mean") < F.col("energy_min"))),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# 9. Features temporelles
df_clean = df_clean \
    .withColumn("year", F.year(F.col("day"))) \
    .withColumn("month", F.month(F.col("day"))) \
    .withColumn("dayofweek", F.dayofweek(F.col("day"))) \
    .withColumn("is_weekend", F.when(F.col("dayofweek").isin([1, 7]), F.lit(1)).otherwise(F.lit(0))) \
    .withColumn("processed_date", F.current_date())

print(f"\nNombre de lignes nettoyées : {df_clean.count()}")
print("\n=== APERÇU NETTOYÉ ===")
df_clean.show(5)

In [6]:
# CELLULE 5 - Statistiques après nettoyage
print("\n=== VALEURS NULLES APRÈS ===")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()

print("\n=== OUTLIERS & CONSISTANCE ===")
df_clean.groupBy("is_outlier_mean").count().show()
df_clean.groupBy("consistency_flag").count().show()

In [7]:
# CELLULE 6 - Sauvegarde Delta
df_clean.write.format("delta").mode("overwrite").save(path_processed)
print(" daily_dataset sauvegardé dans processed/daily_dataset/")

In [8]:
# CELLULE 7 - Vérification
df_verify = spark.read.format("delta").load(path_processed)
print(f"Vérification: {df_verify.count()} lignes")
df_verify.show(5)